# MySong — large-corpus training on Google Colab

Trains the MySong accompaniment model on **5,073 lead sheets** (502 OpenEWLD scores + 4,571 deduplicated AutoHarmonizer/Wikifonia scores) on a Colab GPU, evaluates it on the held-out split, and packages the artifacts for your local checkout.

**What it does, in order**

1. checks the runtime and installs what is missing;
2. gets the project (clone from GitHub, or upload a zip of the folder);
3. builds/verifies the two lead-sheet corpora;
4. verifies that **no held-out melody leaked** into the auxiliary corpus;
5. smoke-tests the trainer on a tiny run (~1 minute) before the long one;
6. trains the 256-unit BiGRU chord tagger on every score, in resumable chunks;
7. optionally trains a second tagger for the two-model ensemble the package supports;
8. evaluates on the held-out test split, runs the package's regression tests, renders a listenable demo, and zips the results.

**Runtime:** `Runtime → Change runtime type → T4 GPU` (or better). The smoke test is about a minute; the full 60-epoch run is roughly 1–3 hours on a T4, about twice that with the ensemble.

**Honesty rules built in:** the OpenEWLD dev/test split is never used for gradients or early stopping, the auxiliary corpus is checked to have zero melody-signature overlap with the held-out split, and the test number at the end is the number you can quote. The reference result for this exact recipe (`docs/mysong.md`, 44 held-out songs) is root 0.592 / core 0.560 / full 0.322 / key 0.977 / changes 0.595.

## How to use this notebook

Run all cells in order (`Runtime → Run all`). Everything is idempotent and safe to re-run:

* the corpora are only rebuilt when they are missing or when you flip `rebuild_base` / `rebuild_extra`;
* training checkpoints every `chunk_epochs` epochs, so if Colab disconnects, run the notebook again from the top and it continues where it stopped;
* set `save_to_drive` to also copy the checkpoints to Google Drive, so they survive even a fresh VM;
* all knobs live in the configuration cell below; the defaults reproduce `GPU_MYSONG.md`.

## 1. Configuration

Everything adjustable lives here. The defaults are the documented recipe: 60 epochs, 256 hidden units, both corpora, plus an optional second seed for the ensemble.

In [ ]:
# === CONFIG ================================================================
CFG = {
    # --- where the project comes from ------------------------------------
    "repo_url": "https://github.com/haaaarsh4/Archer-Robot-Vocal-Collaboration-System.git",
    "branch": "main",
    "repo_dir": "/content/Archer-Robot-Vocal-Collaboration-System",

    # --- environment ------------------------------------------------------
    "allow_cpu": False,           # True only if you accept a many-hour CPU run
    "install_audio_tools": True,  # fluidsynth + General MIDI SoundFont for the listening test

    # --- corpora ----------------------------------------------------------
    "rebuild_base": False,        # re-parse the 502 OpenEWLD scores even if the cache exists
    "rebuild_extra": False,       # re-download and re-build the auxiliary corpus from scratch
    "extra_workers": 2,           # parallel workers for the auxiliary corpus build

    # --- training ---------------------------------------------------------
    "epochs": 60,
    "hidden": 256,
    "chunk_epochs": 10,           # checkpoint/resume every N epochs; 0 = one uninterrupted schedule
    "seed": 0,
    "base_repeat": 1,             # repeat the in-domain OpenEWLD sequences to up-weight them
    "limit_extra": 0,             # 0 = use every auxiliary score
    "ensemble": True,             # also train a second tagger (seed 1) -> 2-model ensemble
    "restart": False,             # ignore existing checkpoints in work_dir and start over
    "init_tagger": None,          # optional .pt tagger checkpoint to fine-tune from

    # --- outputs -----------------------------------------------------------
    "out_model": "data/mysong/model/mysong_large.npz",
    "work_dir": "data/mysong/model/runs",
    "report_dir": "reports/mysong_colab",
    "render_dir": "renders/mysong_demo",
    "promote": False,             # also replace the package defaults (backups are kept)
    "save_to_drive": False,
    "drive_dir": "/content/drive/MyDrive/mysong_large",
}
for key, value in CFG.items():
    print(f"{key:18} {value}")

## 2. Runtime check

Confirms the GPU, memory and disk the rest of the notebook assumes. If there is no GPU it stops here (unless you set `allow_cpu`), because CPU training is many hours instead of minutes.

In [ ]:
# --- runtime check ----------------------------------------------------------
import json
import os
import platform
import shutil
import subprocess
import sys
import time
from pathlib import Path

print("python  ", sys.version.split()[0], "|", platform.platform())
print("cores   ", os.cpu_count())

try:
    import torch
    CFG["torch"] = torch.__version__
    CFG["cuda"] = bool(torch.cuda.is_available())
    CFG["device"] = "cuda" if CFG["cuda"] else "cpu"
    CFG["gpu"] = torch.cuda.get_device_name(0) if CFG["cuda"] else "none"
except Exception as exc:  # noqa: BLE001 - reported, not raised
    CFG["torch"], CFG["cuda"], CFG["device"], CFG["gpu"] = f"missing ({exc})", False, "cpu", "none"

print("torch   ", CFG["torch"])
print("device  ", CFG["device"], "|", CFG["gpu"])
if shutil.which("nvidia-smi"):
    subprocess.run(["nvidia-smi"], check=False)
else:
    print("nvidia-smi not available")

disk = shutil.disk_usage("/")
print(f"disk     {disk.free / 2**30:.1f} GiB free")
try:
    ram = os.sysconf("SC_PAGE_SIZE") * os.sysconf("SC_PHYS_PAGES") / 2**30
    print(f"ram      {ram:.1f} GiB")
except Exception:
    pass
if CFG["device"] == "cuda":
    try:
        free, total = torch.cuda.mem_get_info()
        print(f"vram     {free / 2**30:.1f} GiB free of {total / 2**30:.1f} GiB")
    except Exception:
        pass

if CFG["device"] != "cuda" and not CFG["allow_cpu"]:
    raise SystemExit(
        "No CUDA GPU is visible. In Colab use Runtime -> Change runtime type -> T4 GPU, "
        "then run the notebook again from this cell. Set CFG['allow_cpu'] = True only if "
        "you accept a many-hour CPU run."
    )

## 3. Dependencies

Colab already ships PyTorch with CUDA, numpy, librosa, soundfile and scipy. This cell adds anything missing (`music21` for parsing the corpora, `scikit-learn` for the table-based parts, `pytest` for the regression tests) and installs FluidSynth plus a General MIDI SoundFont so the listening test plays real instruments instead of the built-in fallback synth.

In [ ]:
# --- dependencies -----------------------------------------------------------
import importlib.util

MODULES = [
    ("music21", "music21"),
    ("sklearn", "scikit-learn"),
    ("librosa", "librosa"),
    ("soundfile", "soundfile"),
    ("scipy", "scipy"),
    ("pytest", "pytest"),
]
missing = [package for module, package in MODULES if importlib.util.find_spec(module) is None]
if missing:
    print("installing:", ", ".join(missing))
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)
else:
    print("python dependencies already present")

if CFG["install_audio_tools"]:
    print("installing fluidsynth + General MIDI SoundFont (optional, for the listening test)...")
    subprocess.run(
        "apt-get -qq update && apt-get -qq install -y fluidsynth fluid-soundfont-gm p7zip-full",
        shell=True, check=False,
    )
    print("fluidsynth:", shutil.which("fluidsynth") or "not available (synth fallback will be used)")
    print("7z        :", shutil.which("7z") or shutil.which("7za") or "not available (not required)")

import music21  # noqa: E402 - fail here rather than deep inside the corpus build
print("music21   :", music21.__version__)

## 4. Get the project

Clones the repository at `repo_url`. If the clone fails (private repo, offline VM), the cell asks you to upload a zip of the project folder instead — the upload is unzipped and used as the working copy.

In [ ]:
# --- get the project --------------------------------------------------------
repo = Path(CFG["repo_dir"])


def _clone():
    if repo.exists() and not (repo / "mysong" / "model.py").exists():
        shutil.rmtree(repo)
    if repo.exists():
        return
    print("cloning", CFG["repo_url"])
    result = subprocess.run(
        ["git", "clone", "--depth", "1", "--branch", CFG["branch"], CFG["repo_url"], str(repo)],
        text=True, capture_output=True,
    )
    print((result.stdout or result.stderr).strip()[-2000:])


if not (repo / "mysong" / "model.py").exists():
    _clone()

if not (repo / "mysong" / "model.py").exists():
    # COLAB-ONLY: the clone did not produce the project -> upload a zip of the folder.
    print("Clone failed; upload a zip of the project folder (the folder that contains mysong/).")
    from google.colab import files as _colab_files  # type: ignore
    for name, blob in _colab_files.upload().items():
        archive = Path("/content/_upload.zip")
        archive.write_bytes(blob)
        shutil.unpack_archive(str(archive), "/content/_upload")
        found = next(Path("/content/_upload").rglob("mysong/model.py"), None)
        if found is None:
            raise SystemExit(f"{name} does not contain mysong/model.py")
        source = found.parent.parent
        if source.resolve() != Path(CFG["repo_dir"]).resolve():
            if Path(CFG["repo_dir"]).exists():
                shutil.rmtree(CFG["repo_dir"])
            shutil.move(str(source), CFG["repo_dir"])
            repo = Path(CFG["repo_dir"])
        break

os.chdir(repo)
if str(Path.cwd()) not in sys.path:
    sys.path.insert(0, str(Path.cwd()))
print("working directory:", Path.cwd())

REQUIRED = [
    "mysong/model.py",
    "mysong/neural.py",
    "mysong/train_large.py",
    "mysong/cli.py",
    "tools/train_mysong_tagger_large.py",
    "tools/build_autoharmonizer_corpus.py",
    "data/mysong/corpus.jsonl",
    "data/mysong/openewld/dataset",
]
missing = [item for item in REQUIRED if not Path(item).exists()]
if missing:
    raise SystemExit("the project is incomplete; missing: " + ", ".join(missing))
head = subprocess.run(["git", "rev-parse", "--short", "HEAD"], text=True, capture_output=True).stdout.strip()
print("project ready at commit", head or "(uploaded copy)")

## 5. Base corpus (OpenEWLD) and the structural model

The base corpus is [OpenEWLD](https://github.com/00sapo/OpenEWLD), the public-domain subset of EWLD: 502 lead sheets. The repository ships the parsed cache (`data/mysong/corpus.jsonl`), so this cell only re-parses when the cache is missing (or when `rebuild_base` is on). It also makes sure the structural model (`data/mysong/model/mysong.npz`) exists — the large tagger is attached to it.

In [ ]:
# --- base corpus ------------------------------------------------------------
from mysong.corpus import corpus_stats, load_corpus, split_corpus

base_cache = Path("data/mysong/corpus.jsonl")
if CFG["rebuild_base"] or not base_cache.exists():
    print("parsing OpenEWLD into", base_cache, "(a few minutes)")
    subprocess.run([sys.executable, "-m", "mysong.cli", "corpus"], check=True)

base_songs = load_corpus(base_cache)
stats = corpus_stats(base_songs)
base_split = {name: len(items) for name, items in split_corpus(base_songs).items()}
print(f"base corpus: {stats['songs']} songs, {stats['melody_notes']:,} melody notes, "
      f"{stats['chords']:,} chord spans, {stats['unique_chords']} unique chords")
print("split:", base_split)
if stats["songs"] < 400:
    raise SystemExit("the OpenEWLD cache looks empty; expected about 502 songs")

base_model = Path("data/mysong/model/mysong.npz")
if not base_model.exists():
    print("no structural model yet; fitting it from the base corpus (~1 minute)")
    subprocess.run([sys.executable, "-m", "mysong.cli", "train", "--no-tagger"], check=True)
print(f"structural model: {base_model} ({base_model.stat().st_size / 1024:.0f} KiB)")

## 6. Auxiliary corpus (AutoHarmonizer / Wikifonia, 4,571 scores)

The auxiliary corpus is the biggest quality lever available: the [AutoHarmonizer](https://github.com/sander-wood/autoharmonizer) archive distributes 5,204 Wikifonia lead sheets as MusicXML in a **split ZIP** (`dataset.z01` + `dataset.zip`). Two volumes concatenated in order are exactly the original ZIP, so this cell joins them in Python — no 7-Zip required.

`tools/build_autoharmonizer_corpus.py` then parses every score, removes duplicates by normalized title **and** by transposition-invariant melody signature against the whole OpenEWLD corpus, and writes `data/mysong/corpus_extra.jsonl`. Expected result: **4,571 scores** after 626 duplicates and a handful of unparseable files.

This cell is skipped when the cache is already present and large enough, or when `rebuild_extra` is on.

In [ ]:
# --- auxiliary corpus -------------------------------------------------------
from zipfile import ZipFile

AUTOHARMONIZER = {
    "z01": "https://raw.githubusercontent.com/sander-wood/autoharmonizer/main/dataset/dataset.z01",
    "zip": "https://raw.githubusercontent.com/sander-wood/autoharmonizer/main/dataset/dataset.zip",
}
extra_cache = Path("data/mysong/corpus_extra.jsonl")
archive_root = Path("data/autoharmonizer")
extracted = archive_root / "dataset"


def count_lines(path):
    with Path(path).open(encoding="utf-8") as handle:
        return sum(1 for line in handle if line.strip())


def download(url, dest):
    dest = Path(dest)
    if dest.exists() and dest.stat().st_size > 1_000_000:
        print("already downloaded:", dest)
        return dest
    import urllib.request

    def hook(blocks, block_size, total):
        if total > 0 and blocks % 50 == 0:
            print(f"  {min(100, blocks * block_size * 100 // total)}%", end="\r")

    print("downloading", url)
    urllib.request.urlretrieve(url, dest, hook)
    print(f"saved {dest} ({dest.stat().st_size / 2**20:.1f} MiB)")
    return dest


def extract_archive():
    scores = list(extracted.glob("*.mxl")) + list(extracted.glob("*.musicxml"))
    if len(scores) > 5000:
        print(f"archive already extracted: {len(scores)} scores")
        return
    joined = Path("/tmp/autoharmonizer_joined.zip")
    print("joining the split archive and extracting...")
    with joined.open("wb") as out:
        for part in (Path("/tmp/dataset.z01"), Path("/tmp/dataset.zip")):
            with part.open("rb") as handle:
                shutil.copyfileobj(handle, out, 1 << 20)
    with ZipFile(joined) as archive:
        archive.extractall(archive_root)
    joined.unlink(missing_ok=True)
    print("extracted", len(list(extracted.glob("*.mxl"))), "scores to", extracted)


if CFG["rebuild_extra"] or not (extra_cache.exists() and count_lines(extra_cache) > 4000):
    download(AUTOHARMONIZER["z01"], "/tmp/dataset.z01")
    download(AUTOHARMONIZER["zip"], "/tmp/dataset.zip")
    extract_archive()
    print("building", extra_cache, f"with {CFG['extra_workers']} workers (a few minutes)")
    subprocess.run(
        [sys.executable, "tools/build_autoharmonizer_corpus.py",
         "--root", str(extracted), "--out", str(extra_cache),
         "--openewld", str(base_cache), "--workers", str(CFG["extra_workers"])],
        check=True,
    )
    shutil.rmtree(extracted, ignore_errors=True)  # the cache is what training needs

print("auxiliary corpus:", count_lines(extra_cache), "scores")

## 7. Integrity check — the held-out split must not leak

This is the check that makes the final number honest. The auxiliary archive is Wikifonia-derived, and 40 of the 44 held-out OpenEWLD test tunes were also present in it **under a different title** — title matching alone missed them and produced a fake 0.73 test accuracy. The builder removes duplicates by a transposition-invariant melody signature; this cell independently re-verifies that, and stops before training if anything leaks.

In [ ]:
# --- integrity --------------------------------------------------------------
from mysong.corpus import melody_signature

base_signatures = {
    signature for signature in (melody_signature(song) for song in base_songs)
    if signature is not None
}
splits = split_corpus(base_songs)
held_out = splits["dev"] + splits["test"]
held_out_signatures = {
    signature for signature in (melody_signature(song) for song in held_out)
    if signature is not None
}

extra_songs = load_corpus(extra_cache)
overlap_base = 0
overlap_held_out = 0
for song in extra_songs:
    signature = melody_signature(song)
    if signature is None:
        continue
    if signature in base_signatures:
        overlap_base += 1
    if signature in held_out_signatures:
        overlap_held_out += 1

print(f"auxiliary songs                          : {len(extra_songs)}")
print(f"melody overlap with OpenEWLD (all 502)   : {overlap_base}")
print(f"melody overlap with the held-out dev+test: {overlap_held_out}")
if overlap_base or overlap_held_out:
    raise SystemExit(
        "the auxiliary corpus leaks held-out melodies; rebuild it (CFG['rebuild_extra'] = True) "
        "before training, or the evaluation will be meaningless"
    )
print("integrity check passed: the auxiliary corpus is train-only and the held-out split is clean")

## 8. The trainer (resumable)

This is the same code path as `tools/train_mysong_tagger_large.py`: it streams the auxiliary JSONL one score at a time and keeps only compact per-beat feature/label arrays, so ~5,000 scores fit in about 1.5 GB of RAM. Two additions make it survive Colab:

* training runs in chunks of `chunk_epochs` epochs, each resuming from the previous chunk's best checkpoint, and the state file records how far it got — re-running the notebook continues instead of restarting;
* each chunk's cosine learning-rate schedule restarts (a warm restart), which is the usual way to make long runs resumable. Set `chunk_epochs = 0` for the single uninterrupted schedule from `GPU_MYSONG.md`.

Early stopping always uses a slice of the **training** songs; the dev and test splits are never touched.

In [ ]:
# --- trainer ----------------------------------------------------------------
import shutil
import time

import numpy as np
from mysong.corpus import Song
from mysong.neural import song_sequence, train_tagger_sequences


def build_sequences(extra_path, limit_extra=0, base_repeat=1, seed=0, log=print):
    """Compact (features, labels) sequences: OpenEWLD train plus streamed auxiliary scores."""
    base = load_corpus(base_cache)
    parts = split_corpus(base, seed=seed)
    sequences = []
    for song in parts["train"]:
        item = song_sequence(song)
        if item is not None:
            sequences.append(item)
    sequences *= max(1, int(base_repeat))
    streamed = 0
    path = Path(extra_path)
    if path.exists():
        with path.open(encoding="utf-8") as handle:
            for line in handle:
                if limit_extra and streamed >= int(limit_extra):
                    break
                line = line.strip()
                if not line:
                    continue
                item = song_sequence(Song.from_dict(json.loads(line)))
                if item is not None:
                    sequences.append(item)
                    streamed += 1
                    if streamed % 1000 == 0:
                        log(f"  streamed {streamed} auxiliary scores")
    log(f"sequences: {len(sequences)} (OpenEWLD train {len(parts['train'])} x{base_repeat} "
        f"+ auxiliary {streamed})")
    return parts, sequences, streamed


def drive_backup(path, log=print):
    """Copy a checkpoint to Google Drive when save_to_drive is on."""
    if not CFG["save_to_drive"]:
        return
    try:
        from google.colab import drive  # type: ignore

        if not Path("/content/drive/MyDrive").exists():
            drive.mount("/content/drive")
        target = Path(CFG["drive_dir"])
        target.mkdir(parents=True, exist_ok=True)
        path = Path(path)
        if path.is_dir():
            shutil.copytree(path, target / path.name, dirs_exist_ok=True)
        else:
            shutil.copy2(path, target / path.name)
        log(f"  backed up {path} -> {target}")
    except Exception as exc:  # noqa: BLE001 - a backup must never kill training
        log(f"  drive backup skipped: {exc}")


def train_chunked(name, sequences, epochs, hidden, chunk_epochs, seed, device,
                  init=None, work_dir=None, log=print):
    """Train one tagger in resumable chunks; returns (best .pt, best val root, epochs done)."""
    epochs = int(epochs)
    if epochs < 1:
        raise SystemExit("epochs must be at least 1")
    run_dir = Path(work_dir or CFG["work_dir"]) / name
    run_dir.mkdir(parents=True, exist_ok=True)
    state_path = run_dir / "state.json"
    if CFG["restart"]:
        for item in run_dir.glob("chunk_*.pt*"):
            item.unlink()
        state_path.unlink(missing_ok=True)

    state = json.loads(state_path.read_text()) if state_path.exists() else {}
    done = int(state.get("epochs_done", 0))
    best_root = float(state.get("best_val_root", -1.0))
    best_checkpoint = state.get("best_checkpoint")

    if done >= epochs and best_checkpoint:
        log(f"[{name}] already trained for {done} epochs (best val root {best_root:.3f}); nothing to do")
        return Path(best_checkpoint), best_root, done
    if done == 0 and init:
        init_path = Path(init)
    elif done:
        init_path = run_dir / f"chunk_{done:03d}.pt"
    else:
        init_path = None
    if init_path is not None and not Path(init_path).exists():
        raise SystemExit(f"cannot resume {name}: {init_path} is missing")
    if done:
        log(f"[{name}] resuming from epoch {done} (best val root {best_root:.3f})")

    chunk = int(chunk_epochs) if chunk_epochs else epochs
    while done < epochs:
        take = min(chunk, epochs - done)
        target = run_dir / f"chunk_{done + take:03d}.pt"
        started = time.time()
        report = train_tagger_sequences(
            sequences, target,
            epochs=take,
            hidden=int(hidden),
            seed=int(seed),
            validation_fraction=0.08,
            device=device,
            init_path=init_path,
            log=log,
        )
        done += take
        if float(report.best_val_root) > best_root:
            best_root = float(report.best_val_root)
            best_checkpoint = str(target)
        state_path.write_text(json.dumps({
            "epochs_done": done,
            "epochs_total": epochs,
            "hidden": int(hidden),
            "seed": int(seed),
            "best_val_root": best_root,
            "best_checkpoint": best_checkpoint,
            "updated": time.strftime("%Y-%m-%d %H:%M:%S"),
        }, indent=2), encoding="utf-8")
        elapsed = time.time() - started
        remaining = (epochs - done) * elapsed / max(take, 1)
        log(f"[{name}] epoch {done}/{epochs}  best val root {best_root:.3f}  "
            f"chunk took {elapsed / 60:.1f} min  ~{remaining / 60:.0f} min left")
        drive_backup(run_dir, log=log)
        init_path = target

    return Path(best_checkpoint), best_root, done


def tagger_meta(pt_path, seed=None):
    """Describe one trained tagger from its own files: architecture + run report."""
    import torch

    npz_path = Path(pt_path).with_suffix(".npz")
    with np.load(npz_path, allow_pickle=False) as arrays:
        meta = {
            "hidden": int(arrays["meta:hidden"]),
            "layers": int(arrays["meta:layers"]),
            "parameters": int(sum(int(np.prod(arrays[key].shape)) for key in arrays.files
                                  if not key.startswith("meta:"))),
        }
    report = {}
    try:  # the .pt is optional, and older checkpoints carry no report
        report = torch.load(str(pt_path), map_location="cpu", weights_only=False).get("report", {})
    except Exception:  # noqa: BLE001 - metadata must never break assembly
        pass
    if "best_val_root" in report:
        meta["best_val_root"] = round(float(report["best_val_root"]), 4)
    for key in ("train_songs", "val_songs"):
        if key in report:
            meta[key] = int(report[key])
    if seed is not None:
        meta["seed"] = int(seed)
    return meta


def assemble_model(out_path, tagger_pt, base_model="data/mysong/model/mysong.npz",
                   second=None, epochs=0, extra_scores=0, seed=0, log=print):
    """Copy the structural model and the trained tagger(s) into one loadable checkpoint.

    The structural model carries the metadata of whichever tagger it was fitted
    with, so attaching a new one without rewriting that block would leave
    ``python -m mysong.cli status`` (and the model card) describing the previous
    run.  The block is rebuilt here from the files being attached.
    """
    out = Path(out_path)
    out.parent.mkdir(parents=True, exist_ok=True)
    if Path(base_model).resolve() != out.resolve():
        shutil.copy2(base_model, out)
    pt = Path(tagger_pt)
    shutil.copy2(pt, out.with_name(out.stem + "_tagger.pt"))
    shutil.copy2(pt.with_suffix(".npz"), out.with_name(out.stem + "_tagger.npz"))
    names = [out.name, out.stem + "_tagger.npz"]
    members = [tagger_meta(pt, seed=seed)]
    if second is not None:
        second = Path(second)
        shutil.copy2(second.with_suffix(".npz"), out.with_name(out.stem + "_tagger2.npz"))
        names.append(out.stem + "_tagger2.npz")
        members.append(tagger_meta(second, seed=seed + 1))

    from mysong.model import MySongModel

    model = MySongModel.load(out)
    model.meta.config["tagger"] = {
        "hidden": members[0]["hidden"],
        "layers": members[0]["layers"],
        "parameters": members[0]["parameters"],
        "ensemble": len(members),
        "epochs": int(epochs),
        "extra_scores": int(extra_scores or members[0].get("train_songs", 0)),
        "train_songs": members[0].get("train_songs", 0),
        "val_songs": members[0].get("val_songs", 0),
        "trained": time.strftime("%Y-%m-%d"),
        "members": members,
        "note": ("large-corpus tagger; auxiliary AutoHarmonizer/Wikifonia scores; "
                 "test split has zero melody-signature overlap; early stopping on a "
                 "slice of the training songs, never on dev or test"),
    }
    model.save(out)
    log("assembled " + ", ".join(names))
    return out


## 9. Smoke test

Trains one tiny epoch on a few hundred sequences and scores it on six held-out songs. It proves the whole path — corpus → features → GPU training → checkpoint → `MySongModel.load` → decoder — in about a minute, so a mistake surfaces now instead of after the long run.

In [ ]:
# --- smoke test -------------------------------------------------------------
from mysong.evaluate import chord_accuracy
from mysong.harmonize import Harmonizer, Style
from mysong.melody import Melody
from mysong.model import MySongModel

smoke_dir = Path("/tmp/mysong_smoke")
smoke_split, smoke_sequences, _ = build_sequences(extra_cache, limit_extra=150, base_repeat=1, seed=CFG["seed"])
smoke_pt, smoke_root, _ = train_chunked(
    "smoke", smoke_sequences, epochs=1, hidden=32, chunk_epochs=1,
    seed=CFG["seed"], device=CFG["device"], work_dir=smoke_dir,
)
smoke_path = assemble_model(smoke_dir / "mysong_smoke.npz", smoke_pt)
smoke_model = MySongModel.load(smoke_path)
print(smoke_model.describe())
if smoke_model.tagger is None:
    raise SystemExit("the trained tagger did not load; check the checkpoint names in the run directory")

harmonizer = Harmonizer(smoke_model)
roots = []
for song in smoke_split["test"][:6]:
    harmony = harmonizer.harmonize(
        Melody.from_notes(song.notes, beats_per_bar=song.beats_per_bar),
        key=song.key, style=Style(), beats_per_bar=song.beats_per_bar,
    )
    roots.append(chord_accuracy(song, harmony)["root"])
print(f"smoke root accuracy on 6 held-out songs: {sum(roots) / len(roots):.3f}")
print("smoke test passed — the full run uses the same code path with more epochs and units")

## 10. Full training

Trains the tagger on the OpenEWLD training split **plus every auxiliary score**. With `ensemble` on, it then trains a second tagger with a different seed; `MySongModel.load` averages the two log-probability fields automatically because `mysong_large_tagger2.npz` sits next to the checkpoint.

Two things worth knowing while it runs:

* the log prints one line per epoch; the chunk lines add elapsed time and an estimate of what is left;
* if the session dies, run the notebook again from the top — the corpora checks are cheap and this cell resumes from the last chunk checkpoint (`CFG["restart"] = True` throws those away and starts over).

In [ ]:
# --- full training ----------------------------------------------------------
print(f"training on {CFG['device']} — {CFG['epochs']} epochs, {CFG['hidden']} hidden units, "
      f"{'ensemble of two taggers' if CFG['ensemble'] else 'single tagger'}")

split, sequences, extra_used = build_sequences(
    extra_cache, limit_extra=CFG["limit_extra"], base_repeat=CFG["base_repeat"], seed=CFG["seed"],
)
split_sizes = {name: len(items) for name, items in split.items()}
print("held-out split sizes:", split_sizes)
started = time.time()

tagger_pt, best_root, epochs_done = train_chunked(
    "seed0", sequences,
    epochs=CFG["epochs"], hidden=CFG["hidden"], chunk_epochs=CFG["chunk_epochs"],
    seed=CFG["seed"], device=CFG["device"], init=CFG["init_tagger"],
)
print(f"seed 0: best validation root {best_root:.3f} after {epochs_done} epochs")

second_pt = None
if CFG["ensemble"]:
    print("training the second tagger for the ensemble (same data, different seed)...")
    second_pt, second_root, _ = train_chunked(
        "seed1", sequences,
        epochs=CFG["epochs"], hidden=CFG["hidden"], chunk_epochs=CFG["chunk_epochs"],
        seed=CFG["seed"] + 1, device=CFG["device"], init=None,
    )
    print(f"seed 1: best validation root {second_root:.3f}")

out_model = assemble_model(
    CFG["out_model"], tagger_pt, second=second_pt,
    epochs=CFG["epochs"], extra_scores=extra_used, seed=CFG["seed"],
)
model = MySongModel.load(out_model)
print(model.describe())
print(f"total training time: {(time.time() - started) / 60:.1f} min")

## 11. Optional: make it the package default

`CFG["promote"]` copies the fresh artifacts over `data/mysong/model/mysong.npz` + `mysong_tagger.npz` (the names the app loads), keeping a backup of whatever was there. Leave it off to keep the large model under its own name and use it explicitly.

In [ ]:
# --- promote to the package default -----------------------------------------
if CFG["promote"]:
    out = Path(CFG["out_model"])
    defaults = Path("data/mysong/model")
    backup = defaults / "mysong_pre_large_backup"
    backup.mkdir(exist_ok=True)
    for name in ("mysong.npz", "mysong_tagger.npz", "mysong_tagger.pt", "mysong_tagger2.npz"):
        current = defaults / name
        if current.exists() and not (backup / name).exists():
            shutil.copy2(current, backup / name)
    shutil.copy2(out, defaults / "mysong.npz")
    shutil.copy2(out.with_name(out.stem + "_tagger.npz"), defaults / "mysong_tagger.npz")
    if out.with_name(out.stem + "_tagger.pt").exists():
        shutil.copy2(out.with_name(out.stem + "_tagger.pt"), defaults / "mysong_tagger.pt")
    if out.with_name(out.stem + "_tagger2.npz").exists():
        shutil.copy2(out.with_name(out.stem + "_tagger2.npz"), defaults / "mysong_tagger2.npz")
    print("package defaults now use the freshly trained model; previous files are in", backup)
    print("verify with: python -m mysong.cli status")
else:
    print(f"promote is off — the trained model stays at {out_model} with its tagger files beside it")

## 12. Evaluation on the held-out split

Runs `python -m mysong.cli evaluate` on the dev and test splits. The report includes the model's own numbers plus the paper's fixed-one-chord-per-bar baseline, a key-aware template matcher, and the paper's generative observation with everything else unchanged — the same table as `docs/mysong.md`. The **test** split was never used for gradients or early stopping, so this is the number to quote.

In [ ]:
# --- evaluation -------------------------------------------------------------
report_dir = Path(CFG["report_dir"])
report_dir.mkdir(parents=True, exist_ok=True)
evaluation = {}
for split_name in ("dev", "test"):
    target = report_dir / f"evaluate_{split_name}.json"
    print(f"--- {split_name} ---")
    result = subprocess.run(
        [sys.executable, "-m", "mysong.cli", "evaluate",
         "--model", str(out_model), "--split", split_name, "--output", str(target)],
        text=True, capture_output=True,
    )
    print((result.stdout or result.stderr).strip() or "(no output)")
    if result.returncode != 0:
        raise SystemExit(f"evaluation failed on the {split_name} split")
    evaluation[split_name] = json.loads(target.read_text())

print()
print("reference for this recipe (docs/mysong.md, test split, 44 songs):")
print("  large corpus, auto key   root 0.592  core 0.560  full 0.322  key 0.977  changes 0.595")
print("If your numbers are much lower: check the integrity cell passed, that all epochs ran, "
      "and evaluate once without the ensemble file to see the ablation.")

## 13. Regression tests

Runs the package's own MySong test file (`tests/test_mysong.py`) against the environment you just built: decoder coverage, key classifier accuracy, chunked MIDI writing, the auxiliary-corpus overlap guarantee, and the SoundFont renderer when it is installed.

In [ ]:
# --- regression tests -------------------------------------------------------
# PYTEST_DISABLE_PLUGIN_AUTOLOAD keeps unrelated third-party pytest plugins that
# happen to be installed in the environment out of this run; the test file only
# needs core pytest.
environment = dict(os.environ, PYTEST_DISABLE_PLUGIN_AUTOLOAD="1")
result = subprocess.run(
    [sys.executable, "-m", "pytest", "-q", "tests/test_mysong.py"],
    text=True, capture_output=True, env=environment,
)
print((result.stdout or result.stderr)[-4000:])
if result.returncode != 0:
    raise SystemExit("tests/test_mysong.py failed")
print("all MySong regression tests passed")

## 14. Listening test — the final arbiter

Metrics score chord symbols against one reference; they do not measure whether the accompaniment sounds good. This cell synthesizes an eight-bar sung melody, runs it through the full pipeline (`analyze → harmonize → arrange → mix`), renders with FluidSynth and a real General MIDI bank when available, and plays the mix right here. Listen to it before you trust any number.

In [ ]:
# --- listening test ---------------------------------------------------------
import numpy as np
import soundfile as sf

demo_dir = Path(CFG["render_dir"])
demo_dir.mkdir(parents=True, exist_ok=True)
take_path = demo_dir / "demo_take.wav"

sample_rate, bpm, beats_per_bar = 44100, 96.0, 4.0
seconds_per_beat = 60.0 / bpm
melody_midi = [
    67, 69, 71, 72, 71, 69, 67, 64,
    65, 67, 69, 71, 72, 71, 69, 67,
    71, 72, 74, 76, 74, 72, 71, 69,
    67, 66, 65, 64, 62, 64, 60, 60,
]
audio = np.zeros(int(sample_rate * (len(melody_midi) * seconds_per_beat + 1.0)), dtype=np.float32)
for index, midi in enumerate(melody_midi):
    start = int(index * seconds_per_beat * sample_rate)
    length = int(seconds_per_beat * 0.9 * sample_rate)
    t = np.arange(length, dtype=np.float32) / sample_rate
    frequency = 440.0 * 2 ** ((midi - 69) / 12)
    vibrato = 1.0 + 0.004 * np.sin(2 * np.pi * 5.2 * t)
    phase = 2 * np.pi * np.cumsum(frequency * vibrato) / sample_rate
    wave = np.sin(phase) + 0.35 * np.sin(2 * phase) + 0.12 * np.sin(3 * phase)
    envelope = (np.minimum(1.0, t / 0.04)
                * np.exp(-1.3 * t)
                * np.minimum(1.0, np.maximum(0.0, (length / sample_rate - t) / 0.06)))
    audio[start:start + length] += (wave * envelope * 0.28).astype(np.float32)
peak = float(np.max(np.abs(audio))) or 1.0
sf.write(take_path, audio / peak * 0.7, sample_rate)
print(f"wrote {take_path} ({audio.size / sample_rate:.1f}s of a synthesized sung melody)")

from mysong.accompaniment import soundfont_available

result = subprocess.run(
    [sys.executable, "-m", "mysong.cli", "render", str(take_path),
     "--out", str(demo_dir / "render"), "--style", "jazz",
     "--bpm", str(bpm), "--offset", "0", "--beats-per-bar", str(beats_per_bar)],
    text=True, capture_output=True,
)
print((result.stdout or result.stderr).strip() or "(no output)")
if result.returncode != 0:
    print("the render failed — the trained model is fine; fix the environment (fluidsynth/librosa) to hear it")
else:
    info = json.loads((demo_dir / "render" / "chords.json").read_text())
    print("key:", info["harmony"]["key"], "| confidence:", info["harmony"]["key_confidence"])
    print("roman numerals:", info["harmony"]["roman"])
    print("renderer:", "soundfont" if soundfont_available() else "built-in synth")
    try:
        from IPython.display import Audio, display

        display(Audio(str(demo_dir / "render" / "mix.wav")))
    except Exception as exc:  # noqa: BLE001 - playback is a convenience
        print("cannot embed audio here:", exc)

## 15. Package the artifacts

Zips the trained checkpoints, the evaluation reports and install instructions, copies everything to Google Drive when `save_to_drive` is on, and offers the browser download. The install instructions inside the zip repeat the two file names the package looks for.

In [ ]:
# --- package ----------------------------------------------------------------
out_model = Path(CFG["out_model"])
bundle_dir = Path("/tmp/mysong_bundle")
if bundle_dir.exists():
    shutil.rmtree(bundle_dir)
bundle_dir.mkdir(parents=True)

artifacts = [
    out_model,
    out_model.with_name(out_model.stem + "_tagger.npz"),
    out_model.with_name(out_model.stem + "_tagger.pt"),
    out_model.with_name(out_model.stem + "_tagger2.npz"),
    Path(CFG["report_dir"]) / "evaluate_dev.json",
    Path(CFG["report_dir"]) / "evaluate_test.json",
]
copied = []
for item in artifacts:
    if item.exists():
        shutil.copy2(item, bundle_dir / item.name)
        copied.append(item.name)

readme = bundle_dir / "INSTALL.txt"
readme.write_text(
    "MySong large-corpus artifacts\n"
    "=============================\n\n"
    "Copy these files into data/mysong/model/ in your local checkout:\n\n"
    "    <stem>.npz          the structural model + metadata\n"
    "    <stem>_tagger.npz   the BiGRU tagger trained on the auxiliary corpus\n"
    "    <stem>_tagger2.npz  optional second tagger; loaded as a 2-model ensemble\n"
    "    <stem>_tagger.pt    optional PyTorch copy of the same weights\n\n"
    "Load it explicitly with:\n\n"
    "    from mysong.model import MySongModel\n"
    "    model = MySongModel.load('data/mysong/model/mysong_large.npz')\n\n"
    "MySongModel.load looks for <stem>_tagger.npz / <stem>_tagger.pt next to the\n"
    "checkpoint, so keep the names identical. To make it the app default instead:\n\n"
    "    cp data/mysong/model/mysong_large.npz data/mysong/model/mysong.npz\n"
    "    cp data/mysong/model/mysong_large_tagger.npz data/mysong/model/mysong_tagger.npz\n"
    "    cp data/mysong/model/mysong_large_tagger2.npz data/mysong/model/mysong_tagger2.npz\n",
    encoding="utf-8",
)
zip_path = Path(shutil.make_archive("/tmp/mysong_large_colab", "zip", bundle_dir))
print(f"bundle: {zip_path} ({zip_path.stat().st_size / 2**20:.1f} MiB)")
print("contents:", ", ".join(copied + [readme.name]))

drive_backup(zip_path)
if CFG["save_to_drive"]:
    print("copied to", CFG["drive_dir"])

try:
    from google.colab import files as _colab_files  # type: ignore

    _colab_files.download(str(zip_path))
except Exception as exc:  # noqa: BLE001 - download is a convenience, not the deliverable
    print("browser download unavailable:", exc)
    print("the bundle is at", zip_path)

## 16. Model card and summary

Writes `reports/mysong_colab/model_card.md`: what data went in, how it was split, exactly how the model was trained, and the measured result — the record you would attach to a release.

In [ ]:
# --- model card -------------------------------------------------------------
import numpy as np
import torch

card_path = Path(CFG["report_dir"]) / "model_card.md"
test_ours = evaluation["test"]["auto"]["ours"]
dev_ours = evaluation["dev"]["auto"]["ours"]
test_generative = evaluation["test"].get("generative", {}).get("ours", {})
head = subprocess.run(["git", "rev-parse", "--short", "HEAD"], text=True, capture_output=True).stdout.strip()

# Read the trained tagger itself (not the structural model's own notes): the .pt
# carries the run report, the .npz carries the architecture and every weight.
tagger_npz = out_model.with_name(out_model.stem + "_tagger.npz")
with np.load(tagger_npz, allow_pickle=False) as arrays:
    parameter_count = int(sum(int(np.prod(arrays[key].shape)) for key in arrays.files
                              if not key.startswith("meta:")))
    hidden_units = int(arrays["meta:hidden"])
tagger_report = torch.load(str(out_model.with_name(out_model.stem + "_tagger.pt")),
                           map_location="cpu", weights_only=False).get("report", {})

card = f"""# MySong large-corpus model card

- trained: {time.strftime('%Y-%m-%d %H:%M:%S')} on `{CFG['gpu']}` ({CFG['device']})
- project commit: {head or 'uploaded copy'}
- trainer: `mysong/neural.py::train_tagger_sequences` ({parameter_count:,} parameters,
  {hidden_units} hidden units, {tagger_report.get('train_songs', '?')} training sequences,
  {tagger_report.get('val_songs', '?')} validation sequences)

## Data

| corpus | songs | role |
|---|---:|---|
| OpenEWLD (public-domain EWLD subset) | {stats['songs']} | structural tables + in-domain training; composer-disjoint 80/10/10 split |
| AutoHarmonizer / Wikifonia archive | {len(extra_songs)} | training only, deduplicated by title and transposition-invariant melody signature |

Held-out split sizes: train {split_sizes['train']} / dev {split_sizes['dev']} / test {split_sizes['test']}.
Melody-signature overlap between the auxiliary corpus and the whole OpenEWLD corpus: 0
(verified in this notebook; title matching alone missed 40 of the 44 test tunes).

## Training

- epochs requested: {CFG['epochs']} (chunk size {CFG['chunk_epochs'] or 'single run'}), seed {CFG['seed']}
- auxiliary scores used: {'all ' + str(len(extra_songs)) if not CFG['limit_extra'] else str(CFG['limit_extra'])}
- OpenEWLD sequences repeated {CFG['base_repeat']}x to keep the in-domain corpus from being drowned
- ensemble: {'second tagger, seed ' + str(CFG['seed'] + 1) if CFG['ensemble'] else 'single tagger'}
- early stopping on a slice of the training songs; dev/test never used for gradients or stopping
- best validation root accuracy: {best_root:.3f}

## Held-out result (test split, {test_ours['n']} songs)

| metric | value |
|---|---:|
| root | {test_ours['root']:.3f} |
| core | {test_ours['core']:.3f} |
| full | {test_ours['full']:.3f} |
| key (exact) | {test_ours['key_exact']:.3f} |
| chord changes F1 | {test_ours['change_f1']:.3f} |

Dev split: root {dev_ours['root']:.3f} / core {dev_ours['core']:.3f} / key {dev_ours['key_exact']:.3f}.
The paper's generative observation with everything else unchanged scores root {test_generative.get('root', float('nan')):.3f}
on the same split — the discriminative tagger is what buys the accuracy.

## Limits

- the auxiliary archive is Wikifonia-derived; its files do not carry a blanket open
  licence for the underlying songs, so check before publishing derived weights;
- one global key per song, symbolic supervision, and Western popular/jazz harmony;
- accuracy against one reference is a lower bound on musical quality — listening
  is the arbiter, and a held-out take has been rendered and listened to above.
"""
card_path.write_text(card, encoding="utf-8")

print(card)
print("=" * 72)
print("done.")
print("model     :", out_model)
print("tagger(s) :", out_model.with_name(out_model.stem + "_tagger.npz"),
      "|", out_model.with_name(out_model.stem + "_tagger2.npz") if second_pt else "(single tagger)")
print("reports   :", report_dir)
print("bundle    :", zip_path)

## Troubleshooting

| Symptom | Fix |
|---|---|
| `No CUDA GPU is visible` | `Runtime → Change runtime type → T4 GPU`, then run from the top. CPU training works but takes many hours (`CFG["allow_cpu"] = True`). |
| Clone failed | The repo is private or the VM has no network: let the upload cell take a zip of the project folder instead. |
| Session disconnected mid-training | Run the notebook again from the top. The corpus checks are cheap and training resumes from `CFG["work_dir"]`; turn on `save_to_drive` so checkpoints survive a brand-new VM. |
| Test numbers below the reference | Confirm the integrity cell printed 0 overlaps, that all epochs ran, and evaluate once without `mysong_large_tagger2.npz` to see the ensemble's contribution. |
| No sound in the listening test | FluidSynth or the SoundFont is missing; the renderer falls back to the built-in synth, which sounds much simpler. |

**Next steps after this notebook:** commit the retrained artifacts with the metrics and the model card, and re-run `python -m mysong.cli status` locally to confirm the app picks them up.